# Example 1 / 1: A Complete Model

Companion notebook for **1 Model Components** in the AMPL lab Documentation, section "A Complete Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Complete Model

All five components together, in the production model used throughout this chapter: how many
tons of each product to make, to earn the most profit with the hours available.

The model is written in AMPL (`model.mod`); the data and the solve come from Python, with amplpy
(`data.py`, `run.py`). In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already
read `model.mod`; the section *Running the examples* below shows the whole program.
`ampl.set["PROD"]` and `ampl.param["rate"]` give the set and the parameters their values: a list
for a set, a number for a scalar parameter, a dictionary keyed by the set's members for an
indexed one. `ampl.solve(solver="highs")` sends the problem to the HiGHS solver,
`ampl.solve_result` says how the solve ended, and `ampl.var["Make"].to_pandas()` returns the
variable's values as a pandas DataFrame (its column `Make.val` holds the values):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                       # products

param rate {PROD} > 0;          # tons produced per hour
param profit {PROD};            # profit per ton
param market {PROD} >= 0;       # most that can be sold, in tons
param avail >= 0;               # hours available

var Make {p in PROD} >= 0, <= market[p];   # tons to make

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];

subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Make"].to_pandas())

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solved
       Make.val
bands      6000
coils      1400


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The data file gives the set and the parameters,
`option solver highs;` chooses the solver, `solve;` runs it, and `display` prints the results:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result;
display Make;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
solve_result = solved

Make [*] :=
bands  6000
coils  1400
;

